# Oceanography: from profiles to an interactive tool

**Faculty workshop · 15–20 minute demonstration · Python 3.10+ · CPU only**

Run this notebook in **CloudBank Classroom JupyterHub** with the complete `workshop` folder uploaded. Open `notebooks/02_oceanography.ipynb` and select **Run → Run All Cells**. You can also run it in a local Jupyter session or Google Colab; a standalone notebook downloads the small public source file if the bundled CSV is absent.

You will read real ocean observations, check their quality flags, visualize temperature and salinity, and use dropdowns to explore a reusable chart function. No AI account or API key is required to run the notebook. When you use an AI assistant to modify code, inspect the change and run it before trusting the interpretation.

**Questions to investigate**

1. How do temperature and practical salinity vary with sea-water pressure in these three casts?
2. How does the difference between the upper 2–50 dbar and 100–150 dbar temperature bands vary by station?
3. What changes when we include interpolated measurements or compare casts with different vertical coverage?

The observations come from the **GO-SHIP P02W 2022 cruise**, R/V Roger Revelle. This is a small cross-section of observations at different places and dates, so these plots cannot establish a long-term warming trend or isolate a cause.

## 1. Set up the notebook

The setup installs a package only if it is missing from the current kernel. A facilitator should run this before class. If your JupyterHub disables package installation, ask its administrator to provide `numpy`, `pandas`, `matplotlib`, and `ipywidgets`; the bundled observations need no network connection.

In [ ]:
import importlib.util
import subprocess
import sys

needed = {"numpy": "numpy>=1.24", "pandas": "pandas>=1.5", "matplotlib": "matplotlib>=3.7", "ipywidgets": "ipywidgets>=8.0"}
missing = [package for module, package in needed.items() if importlib.util.find_spec(module) is None]
INSTALL_MISSING = False  # instructor setup only, if permitted by the classroom
print("Missing packages:", missing or "none")
if missing and INSTALL_MISSING:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
elif missing:
    raise ImportError("Prepare the missing packages using CLOUDBANK_SETUP.md, then restart the kernel.")

from pathlib import Path
from urllib.request import urlopen
from urllib.parse import quote
import hashlib
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
print(f"Python {sys.version.split()[0]} | pandas {pd.__version__} | numpy {np.__version__}")

## 2. Load a small, reproducible subset

The bundled file contains **682 observations** from stations **3, 50, and 100**, cast 1, at pressures up to 600 dbar. Station 3 was shallower and ends at 158 dbar. A row is a pressure level within a cast; it is **not an independent replicate of an ocean location**.

**Source and reuse:** [CCHDO cruise page](https://cchdo.ucsd.edu/cruise/33RR20220430); [NOAA PMEL ERDDAP metadata, units, QC, and CC0 declaration](https://data.pmel.noaa.gov/generic/erddap/info/cchdo_ctd/index.html). Data are offered under [CC0-1.0](https://creativecommons.org/publicdomain/zero/1.0/) by the NOAA endpoint. Credit CCHDO, GO-SHIP P02W investigators, and NOAA PMEL ERDDAP. Source checked on **October 13, 2026**. The cruise page records reprocessing; a fresh download could therefore differ from the supplied cache.

| Column | Meaning | Unit / convention |
|---|---|---|
| `expocode` | Cruise identifier | text |
| `station`, `cast`, `profile_id` | Sampling identifiers | text / integer |
| `time` | Cast timestamp | UTC |
| `latitude`, `longitude` | Cast location | degrees north / east |
| `pressure` | Sea-water pressure | dbar, positive downward |
| `ctd_temperature` | In-situ water temperature | °C, ITS-90 |
| `ctd_salinity` | Practical salinity | dimensionless, PSS-78 |
| `ctd_temperature_qc`, `ctd_salinity_qc` | Instrument quality flags | WOCE CTD codes |

**Pressure is not depth.** Although dbar and metres are often numerically close near the surface, this notebook plots the measured pressure in dbar and does not relabel it as depth. Practical salinity is dimensionless; it is not a concentration in g/kg.

The original ERDDAP CSV includes a second row of units. We skip that row when reading observations.

In [ ]:
QUERY = (
    'expocode,station,cast,profile_id,time,latitude,longitude,pressure,'
    'ctd_temperature,ctd_temperature_qc,ctd_salinity,ctd_salinity_qc'
    '&expocode="33RR20220430"&station=~"3|50|100"&cast=1&pressure<=600'
)
SOURCE_URL = "https://data.pmel.noaa.gov/generic/erddap/tabledap/cchdo_ctd.csv?" + quote(QUERY, safe=",&=")
CACHE_NAME = "p02w_upper600_erddap.csv"
BUNDLED_SHA256 = "8edc63a57d031cff681540033ff3efb29fca727f01504db880adb6f8f49450ea"

candidates = [
    Path.cwd() / "../data/oceanography" / CACHE_NAME,   # notebook opened in workshop/notebooks
    Path.cwd() / "data/oceanography" / CACHE_NAME,      # working directory is workshop
    Path.cwd() / "workshop/data/oceanography" / CACHE_NAME, # working directory is project root
    Path.cwd() / CACHE_NAME,                           # notebook + CSV uploaded together
]
DATA_PATH = next((p.resolve() for p in candidates if p.is_file()), None)
if DATA_PATH is None:
    DATA_PATH = Path.cwd() / "workshop_data/oceanography" / CACHE_NAME
    DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
    try:
        with urlopen(SOURCE_URL, timeout=45) as response:
            source_bytes = response.read()
        if not source_bytes.startswith(b"expocode,station,cast,"):
            raise ValueError("The source did not return the expected CSV table.")
        DATA_PATH.write_bytes(source_bytes)
        print("Downloaded the public NOAA subset. Upload the bundled CSV for offline use.")
    except Exception as exc:
        raise RuntimeError(
            "Could not obtain the source data. Upload p02w_upper600_erddap.csv next to this notebook "
            "or upload the full workshop folder, then rerun this cell."
        ) from exc
else:
    print("Using bundled/local data:", DATA_PATH)

sha256 = hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()
print("Matches workshop snapshot:", sha256 == BUNDLED_SHA256)
if sha256 != BUNDLED_SHA256:
    print("The source file differs from the workshop snapshot; check metadata and row counts before comparing results.")

raw = pd.read_csv(DATA_PATH, skiprows=[1], dtype={"station": str, "profile_id": str})
raw["time"] = pd.to_datetime(raw["time"], utc=True)
required = {"profile_id", "station", "cast", "time", "latitude", "longitude", "pressure", "ctd_temperature", "ctd_salinity", "ctd_temperature_qc", "ctd_salinity_qc"}
if not required.issubset(raw.columns):
    raise ValueError("Unexpected data columns; inspect the source file before continuing.")
print(f"{len(raw):,} rows; {raw.profile_id.nunique()} casts; {raw.station.nunique()} stations")
display(raw.head())

## 3. Quality flags and coverage are part of the result

The source uses WOCE **CTD instrument** flags: **2 = acceptable measurement**, **3 = questionable**, **4 = bad**, and **6 = interpolated over a pressure interval larger than 2 dbar**. Other codes describe uncalibrated, absent, despiked, or unflagged data. CTD flags differ from some bottle-data conventions, so look up the relevant convention each time.

Our teaching policy keeps rows with **both flags equal to 2**, finite values, and plausible broad ranges. The range check supplements the source QC; it cannot certify instrument accuracy. We report exclusions rather than silently hiding them. In the supplied file, three interpolated surface rows carry flag 6.

**Discuss:** Would flag 6 be useful for a continuous display? Would you include it when making a claim about near-surface measurements?

In [ ]:
measurement_columns = ["pressure", "ctd_temperature", "ctd_salinity"]
flag_counts = raw.groupby(["ctd_temperature_qc", "ctd_salinity_qc"]).size().reset_index(name="rows")
display(flag_counts)

range_ok = (
    np.isfinite(raw[measurement_columns]).all(axis=1)
    & raw["pressure"].between(0, 600)
    & raw["ctd_temperature"].between(-3, 40)
    & raw["ctd_salinity"].between(0, 45)
)
quality_ok = (raw["ctd_temperature_qc"] == 2) & (raw["ctd_salinity_qc"] == 2)
clean = raw.loc[range_ok & quality_ok].copy().sort_values(["station", "pressure"])

print(f"Rows retained: {len(clean)} / {len(raw)}")
print(f"Rows failing broad range/finite-value check: {(~range_ok).sum()}")
print(f"Rows excluded by strict flag policy among range-valid rows: {(range_ok & ~quality_ok).sum()}")
display(raw.loc[~(range_ok & quality_ok), ["station", "pressure", "ctd_temperature_qc", "ctd_salinity_qc"]])

coverage = clean.groupby("station", sort=False).agg(
    observations=("pressure", "size"),
    minimum_pressure_dbar=("pressure", "min"),
    maximum_pressure_dbar=("pressure", "max"),
    timestamp_UTC=("time", "first"),
    latitude_degrees_north=("latitude", "first"),
    longitude_degrees_east=("longitude", "first"),
)
display(coverage)
if clean.empty or clean.duplicated(["profile_id", "pressure"]).any():
    raise ValueError("Empty data or repeated pressure levels: investigate before plotting.")

## 4. Draw two linked profiles

A deeper pressure level appears lower on the page. Each line joins samples from **one cast**; the three lines do not describe changes through time at a fixed station. The labels show the station and date so this distinction is visible.

In [ ]:
STATIONS = ["3", "50", "100"]
COLORS = {"3": "#0072B2", "50": "#D55E00", "100": "#009E73"}

fig, axes = plt.subplots(1, 2, figsize=(11, 5), sharey=True, constrained_layout=True)
for station in STATIONS:
    profile = clean.loc[clean["station"] == station].sort_values("pressure")
    date = profile["time"].iloc[0].strftime("%Y-%m-%d")
    label = f"Station {station} · {date}"
    axes[0].plot(profile["ctd_temperature"], profile["pressure"], color=COLORS[station], label=label)
    axes[1].plot(profile["ctd_salinity"], profile["pressure"], color=COLORS[station], label=label)
axes[0].set(xlabel="In-situ temperature (°C; ITS-90)", ylabel="Sea-water pressure (dbar)")
axes[1].set(xlabel="Practical salinity (PSS-78; dimensionless)")
axes[0].invert_yaxis()
for ax in axes:
    ax.grid(alpha=0.25)
axes[1].legend(loc="lower left", fontsize=8)
fig.suptitle("P02W 2022: three casts, with acceptable CTD flags only")
plt.show()

## 5. Ask a numerical question with a common pressure band

To compare upper and lower sampled water, use 2–50 dbar and 100–150 dbar: **all three casts cover these bands**. The function reports how many samples contribute and refuses incomplete coverage. The boundary rule is inclusive. These are arithmetic means of accepted pressure-level observations, not spatial ocean averages; this subset has a 2 dbar sampling grid.

The reported difference is **lower-band mean minus upper-band mean**. A negative difference describes cooler water at higher pressure in that cast. It cannot identify the cause.

In [ ]:
def band_summary(frame, low=100, high=150):
    """Summarize a common pressure band, returning coverage and sample count."""
    if not (0 <= low < high <= 600):
        raise ValueError("Choose 0 <= low < high <= 600 dbar.")
    rows = []
    for station in STATIONS:
        profile = frame.loc[frame["station"] == station]
        available = not profile.empty and profile.pressure.min() <= low and profile.pressure.max() >= high
        band = profile.loc[profile["pressure"].between(low, high)] if available else profile.iloc[:0]
        rows.append({
            "station": station,
            "complete_band_coverage": available,
            "samples": len(band),
            "mean_temperature_C": band["ctd_temperature"].mean(),
            "mean_practical_salinity": band["ctd_salinity"].mean(),
        })
    return pd.DataFrame(rows).set_index("station")

upper = band_summary(clean, 2, 50)
lower = band_summary(clean, 100, 150)
comparison = lower[["complete_band_coverage", "samples"]].rename(columns={"samples": "lower_band_samples"})
comparison["upper_band_samples"] = upper["samples"]
comparison["upper_mean_C"] = upper["mean_temperature_C"]
comparison["lower_mean_C"] = lower["mean_temperature_C"]
comparison["lower_minus_upper_C"] = comparison["lower_mean_C"] - comparison["upper_mean_C"]
display(comparison.round(3))

# Coverage matters: the shallow station is reported as unavailable, not extrapolated.
display(band_summary(clean, 500, 600).round(3))

## 6. Build an interactive analysis tool

A function is a reusable tool with explicit inputs. Here, `plot_profile` accepts a station, variable, pressure limit, and interpolation policy. The dropdowns call the same function that a researcher could use in a script. No web server, deployment, or model is required.

The optional median smooth is a display aid over **11 accepted observations** (about 20 dbar on this sampling grid). It can hide sharp features. Start with raw observations and inspect the smooth separately.

In [ ]:
VARIABLES = {
    "Temperature": ("ctd_temperature", "In-situ temperature (°C; ITS-90)"),
    "Practical salinity": ("ctd_salinity", "Practical salinity (PSS-78; dimensionless)"),
}

def plot_profile(station="50", variable="Temperature", max_pressure=300, include_interpolated=False, smooth=False):
    """Plot one cast. Filtering uses both source flags and keeps units explicit."""
    if str(station) not in STATIONS or variable not in VARIABLES:
        raise ValueError("Choose a known station and variable.")
    if not 10 <= max_pressure <= 600:
        raise ValueError("max_pressure must be between 10 and 600 dbar.")
    allowed_flags = [2, 6] if include_interpolated else [2]
    policy = raw["ctd_temperature_qc"].isin(allowed_flags) & raw["ctd_salinity_qc"].isin(allowed_flags)
    profile = raw.loc[range_ok & policy & (raw["station"] == str(station)) & (raw["pressure"] <= max_pressure)].sort_values("pressure")
    if profile.empty:
        print("No observations meet these settings.")
        return None
    column, label = VARIABLES[variable]
    fig, ax = plt.subplots(figsize=(5.6, 4.5), constrained_layout=True)
    ax.plot(profile[column], profile["pressure"], color=COLORS[str(station)], marker=".", markersize=3, linewidth=1, label="Source values")
    if smooth:
        median = profile[column].rolling(window=11, center=True, min_periods=1).median()
        ax.plot(median, profile["pressure"], color="black", linewidth=2, label="11-observation median")
        ax.legend(fontsize=8)
    date = profile["time"].iloc[0].strftime("%Y-%m-%d")
    ax.set(xlabel=label, ylabel="Sea-water pressure (dbar)", title=f"Station {station} · {date} · n={len(profile)}")
    ax.set_ylim(max_pressure, 0)
    ax.grid(alpha=0.25)
    plt.show()
    print(f"Observed range shown: {profile.pressure.min():g}–{profile.pressure.max():g} dbar; accepted QC flags {allowed_flags}.")
    if profile.pressure.max() < max_pressure:
        print("The empty lower part of the chart shows absent coverage; no values were extrapolated.")
    return fig

# A direct call works even if your notebook frontend does not display widgets.
example_figure = plot_profile(station="50", variable="Temperature", max_pressure=300)

In [ ]:
# Colab may need its widget manager; CloudBank/local Jupyter do not.
if "google.colab" in sys.modules:
    from google.colab import output
    output.enable_custom_widget_manager()

controls = {
    "station": widgets.Dropdown(options=STATIONS, value="50", description="Station"),
    "variable": widgets.Dropdown(options=list(VARIABLES), value="Temperature", description="Variable"),
    "max_pressure": widgets.IntSlider(min=20, max=600, step=20, value=300, description="Max dbar", continuous_update=False),
    "include_interpolated": widgets.Checkbox(value=False, description="Include QC 6"),
    "smooth": widgets.Checkbox(value=False, description="Median smooth"),
}
output_panel = widgets.interactive_output(plot_profile, controls)
display(widgets.VBox(list(controls.values())), output_panel)

## 7. Use AI to make a targeted change

Work in pairs. Copy a prompt below into your institution-approved AI assistant with the relevant function or cell. Read its proposed edits; decide whether the units and QC policy are preserved, then run the revised code in a **new cell**. Keep the original function for comparison.

**Create a chart**

> I teach oceanography and am new to Python. Using the `clean` pandas table in this notebook, write one cell that plots temperature versus practical salinity for each station, with three colorblind-friendly colors and a legend showing dates. Temperature is in-situ °C ITS-90 and practical salinity is dimensionless PSS-78. Do not calculate potential temperature, density, or assign named water masses. Explain the figure in three plain sentences and identify one limitation.

**Extend the tool**

> Modify `plot_profile` to offer a checkbox for showing both temperature and salinity as two panels with a shared pressure axis. Preserve both source QC checks, show units, invert the pressure axis only once, and do not extrapolate. Give me the replacement function and widget cell, and explain where to paste them.

**Check the interpretation**

> Review this proposed claim: “The Pacific warmed from 20.5°C to 22.5°C during this cruise.” The notebook contains three casts at different longitudes and dates in May–June 2022. Identify the logical error, describe a narrower observation the data can support, and suggest the additional observations needed to test a warming trend. Do not invent numbers or cite papers you have not checked.

**Challenge prompt**

> Add a button that exports the selected chart as a PNG and records station, date, variable, QC policy, and pressure cutoff in a companion JSON file. Use the notebook's output directory. Keep all source data unchanged and explain how I can verify the exported metadata.

An accurate AI answer should preserve measurement scales and source flags, show vertical coverage, and avoid treating hundreds of pressure levels as hundreds of independent station samples.

## 8. Save reviewable outputs

This cell exports the original comparison figure, accepted observations, and numerical comparison. It also records the source-file hash and filtering policy. Download these files from the Jupyter file browser if you want to use them in teaching slides.

In [ ]:
if DATA_PATH.parent.name == "oceanography" and DATA_PATH.parent.parent.name == "data":
    OUTPUT_DIR = DATA_PATH.parent.parent.parent / "outputs/oceanography"
else:
    OUTPUT_DIR = Path.cwd() / "workshop_outputs/oceanography"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
fig.savefig(OUTPUT_DIR / "temperature_salinity_profiles.png", dpi=160, bbox_inches="tight")
clean.to_csv(OUTPUT_DIR / "accepted_observations.csv", index=False)
comparison.to_csv(OUTPUT_DIR / "temperature_band_comparison.csv")
run_metadata = {
    "source_url": SOURCE_URL,
    "cruise_url": "https://cchdo.ucsd.edu/cruise/33RR20220430",
    "source_sha256": sha256,
    "source_rows": len(raw),
    "accepted_rows": len(clean),
    "qc_policy": "Temperature and salinity WOCE CTD flag 2 only, finite values and broad range checks",
    "pressure_unit": "dbar; measured pressure, not derived depth",
    "temperature_scale": "ITS-90 degrees Celsius, in-situ",
    "salinity_scale": "PSS-78 practical salinity, dimensionless",
    "comparison_bands_dbar": {"upper": [2, 50], "lower": [100, 150]},
    "limitations": "Different locations and dates, one cast per station, incomplete vertical coverage; no long-term trend inference",
}
(OUTPUT_DIR / "analysis_metadata.json").write_text(json.dumps(run_metadata, indent=2) + "\n")
print("Saved files in", OUTPUT_DIR.resolve())
for path in sorted(OUTPUT_DIR.iterdir()):
    print(" -", path.name)

## Debrief

What did the AI help you change? Which claim or implementation detail needed human review? What would you add before using this tool in a research workflow?

For a second class session, substitute a [small Argo profile subset](https://www.aoml.noaa.gov/proj/argo/) to teach delayed-mode quality control, or a [NOAA OISST subset](https://www.ncei.noaa.gov/products/optimum-interpolation-sst) to explore daily temperature anomalies. Argo profiles and gridded, interpolated OISST fields represent different kinds of observations; adapt the question and quality policy to the source.